In [ ]:
import pandas as pd
import re

# Load the dataset
df = pd.read_csv("clef_taskC_test3.csv")

def accurate_split_note(note):
    """
    Split a note into Subjective, Objective, Assessment, and Plan:
      - SUBJECTIVE: From "CHIEF COMPLAINT"/"CC:" up to before any objective marker
      - OBJECTIVE: From first objective marker to before "ASSESSMENT"/"IMPRESSION"
      - ASSESSMENT: From "ASSESSMENT"/"IMPRESSION" to "PLAN", if found
      - PLAN: From "PLAN" to end
    """
    note = note.replace('\r\n', '\n').replace('\r', '\n').strip()

    # SUBJECTIVE
    cc_start = note.find("CHIEF COMPLAINT")
    if cc_start == -1:
        cc_start = note.find("CC:")
    if cc_start == -1:
        cc_start = 0  # fallback to start

    possible_obj_markers = ["PHYSICAL EXAMINATION", "EXAM", "RESULTS", "PHYSICAL EXAM", "ASSESSMENT", "IMPRESSION", "VITALS"]
    obj_start_candidates = [note.find(marker, cc_start) for marker in possible_obj_markers if note.find(marker, cc_start) != -1]
    first_obj_index = min(obj_start_candidates) if obj_start_candidates else -1

    if cc_start != -1 and first_obj_index != -1:
        subjective = note[cc_start:first_obj_index].strip()
    else:
        subjective = note[cc_start:].strip()

    # OBJECTIVE
    assessment_markers = ["ASSESSMENT AND PLAN", "ASSESSMENT", "IMPRESSION"]
    assessment_start_candidates = [note.find(marker, first_obj_index) for marker in assessment_markers if note.find(marker, first_obj_index) != -1]
    assessment_start = min(assessment_start_candidates) if assessment_start_candidates else -1

    if first_obj_index != -1 and assessment_start != -1:
        objective = note[first_obj_index:assessment_start].strip()
    else:
        objective = ""

    # ASSESSMENT and PLAN using headers if present (case-sensitive)
    assessment_header_match = re.search(r'\n(ASSESSMENT|IMPRESSION)\s*\n', note)
    plan_header_match = re.search(r'\nPLAN\s*\n', note)

    if assessment_header_match and plan_header_match and assessment_header_match.start() < plan_header_match.start():
        assessment = note[assessment_header_match.start():plan_header_match.start()].strip()
        plan = note[plan_header_match.start():].strip()
    else:
        # Fallback parsing using diagnosis blocks
        if assessment_start != -1:
            ass_plan_block = note[assessment_start:].strip()
        else:
            ass_plan_block = ""

        blocks = re.split(r'\n(?=\d+\.\s)', ass_plan_block)
        assessment_list = []
        plan_list = []

        for block in blocks:
            diagnosis_match = re.match(r"(\d+\.\s.+?)\n", block)
            diagnosis_title = diagnosis_match.group(1).strip() if diagnosis_match else "Diagnosis"

            reasoning_matches = re.findall(r"(?i)medical reasoning:\s*(.*?)(?=\n-|\n\d+\.|$)", block, re.DOTALL)
            for reasoning in reasoning_matches:
                assessment_list.append(f"{diagnosis_title}\n- Medical Reasoning: {reasoning.strip()}")

            plan_components = re.findall(
                r"(?i)(Patient Education and Counseling|Medical Treatment|Additional Testing|Specialist Referrals|Patient Agreements|Instructions):([\s\S]*?)(?=\n-|\n\d+\.|$)",
                block
            )
            for header, body in plan_components:
                plan_list.append(f"- {header.strip()}: {body.strip()}")

        assessment = "\n\n".join(assessment_list).strip()
        plan = "\n\n".join(plan_list).strip()

    return subjective, objective, assessment, plan

# Process each note
output_rows = []

for _, row in df.iterrows():
    encounter_id = row['encounter_id']
    note = row['note']

    subj, obj, assess, plan = accurate_split_note(note)

    sections = [("SUBJECTIVE", subj), ("OBJECTIVE", obj), ("ASSESSMENT", assess), ("PLAN", plan)]
    for idx, (section_name, content) in enumerate(sections):
        output_rows.append({
            "encounter_id": encounter_id,
            "section": section_name,
            "content": content,
            "section_encounter_id": f"{encounter_id}_{idx}"
        })

# Save to DataFrame and CSV
soap_df = pd.DataFrame(output_rows)
soap_df.to_csv("test3_section.csv", index=False)

print("✅ Structured SOAP notes saved to 'soap_structured_notes_final_01.11.csv'")
